# Tutorial 4: Neural Networks in Practice — Predicting Toxicity with Tox21

This notebook is the hands-on companion to **Neural Networks and Deep Learning**.

## Learning goals
- Get to know the **Tox21** toxicity dataset and why its labels are sparse
- Visualize a molecular featurization (Morgan fingerprints) and sanity-check it before training on it
- Build an MLP from scratch in PyTorch: forward pass, loss, backpropagation, optimizer updates
- Train a baseline model on one Tox21 task, recognize overfitting in its learning curve, and fight it with regularization (early stopping, weight decay, dropout)
- Question whether a bigger model is justified by the dataset size
- Compare learning curves across hyperparameters and run a hyperparameter search
- Use a **fixed train/val/test split** for fair comparisons, and export a standardized payload for the class Pareto-front analysis
- **Advanced**: extend the model to multi-task learning across several Tox21 assays at once

The notebook is organized as a series of exercises. Each exercise states a short task.

**This is the exercise notebook** — the code cells after each exercise are left blank (marked `# your code here!`) for you to fill in, following the hints given. A fully worked `deep-learning-neural-network_solutions.ipynb` is also available if you get stuck or want to check your answer.

---
## References & tools used

This notebook builds on and credits:

- **Tox21 dataset** — a US EPA/NIH/FDA collaborative screening of ~8,000 compounds against 12 nuclear-receptor and stress-response assays (the "Tox21 Data Challenge", 2014).
- **DeepChem** (`dc.molnet.load_tox21`) — used once, offline, in `neural-networks-mlp/build_dataset.py` to fetch and flatten the raw Tox21 labels into `dataset.csv`. Students don't need DeepChem installed to run this notebook. Ramsundar et al., *Deep Learning for the Life Sciences*, O'Reilly, 2019 — https://deepchem.io
- **RDKit** — used throughout for parsing SMILES and computing Morgan/ECFP fingerprints — https://www.rdkit.org
- **scikit-learn** — PCA, k-NN, feature scaling, ROC-AUC / ROC-curve metrics.
- **PyTorch** — the MLP model, training loop, and autograd.
- **`molecular-representations.ipynb`** (this bootcamp, Tutorial 1) — the `morgan_matrix` fingerprint helper and the chemical-space PCA visualization pattern used in Exercise 2 are adapted from that notebook's featurization sections.
- **[dmol.pub — *Deep Learning for Molecules and Materials*](https://dmol.pub/dl/)** — Section C.6 (introduction.html) inspires the simplest-possible-network example in Exercise 3a; Section 7.6 (layers.html), an overview of regularization techniques, inspires Exercise 5.
- **Kaggle — ["Molecular Toxicity Prediction using ML"](https://www.kaggle.com/code/gyanendrachaubey/molecular-toxicity-prediction-using-ml)** by gyanendrachaubey — inspiration for the missing-label heatmap (Exercise 1) and the ROC-curve evaluation plot (Exercise 9), both common EDA/evaluation views for Tox21-style multi-assay toxicity data.

In [ ]:
# --- Cell: dependency installation ---
# If needed, install common dependencies.
# In most Colab runtimes these are already available; running this is a no-op there.
# If you're running locally, make sure you've created and activated the conda
# environment described in environment.yml (repo root) instead of relying on this line.
# %pip install -q pandas numpy matplotlib scikit-learn torch rdkit

In [ ]:
# --- Cell: imports, random seed, and compute device ---
# Everything used later in the notebook is imported here in one place:
# - stdlib utilities (json/time/random/dataclasses/datetime/pathlib) for config,
#   timing, and writing the leaderboard payload
# - numpy/pandas for data handling, matplotlib for plots
# - RDKit for turning SMILES into Morgan fingerprints (Exercise 2)
# - scikit-learn for PCA/k-NN/scaling/metrics
# - torch for building and training the neural network

import json
import time
import random
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from rdkit import Chem, DataStructs
from rdkit.Chem import rdFingerprintGenerator, Draw

from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import roc_auc_score, roc_curve
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Fix every source of randomness (Python, NumPy, PyTorch) so that re-running this
# notebook reproduces the same weight initialization, batch shuffling, etc.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Use a GPU automatically if one is available (e.g. on Colab), otherwise fall back to CPU.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

In [ ]:
# --- Cell: locate the tutorial data, whether running on Colab or locally ---
# This notebook needs to find the `neural-networks-mlp/` data folder (dataset.csv +
# splits/ + leaderboard/). Its location differs between environments:
# - Locally: you're expected to launch Jupyter from inside `tutorials/`, so it's
#   already in the current working directory.
# - On Colab: nothing is checked out by default, so we clone the bootcamp repo
#   into /content and point the working directory there.
# The logic below tries a few known locations first, and only clones the repo
# (network operation) as a last resort when running on Colab.
import os
import sys
from pathlib import Path

REQUIRED_DATA_REL = Path("neural-networks-mlp/dataset.csv")
CANDIDATE_REPO_URLS = [
    "https://github.com/truejulosdu13/ai4chemistry-bootcamp.git",
    "https://github.com/julschleinitz/ai4chemistry-bootcamp.git",
]

candidate_tutorial_dirs = [
    Path.cwd(),
    Path("/content/ai4chemistry-bootcamp/tutorials"),
    Path("/content/ai4chemistry-bootcamp/website/tutorials"),
]

def _find_tutorial_dir(candidates):
    """Return the first candidate directory that already contains the required
    dataset file, or None if none of them do."""
    for d in candidates:
        if (d / REQUIRED_DATA_REL).exists():
            return d
    return None

tutorial_dir = _find_tutorial_dir(candidate_tutorial_dirs)

# If not found and running in Colab, clone automatically.
if tutorial_dir is None and "google.colab" in sys.modules:
    clone_root = Path("/content/ai4chemistry-bootcamp")
    if not clone_root.exists():
        clone_ok = False
        for repo_url in CANDIDATE_REPO_URLS:
            exit_code = os.system(f"git clone {repo_url} {clone_root}")
            if exit_code == 0:
                clone_ok = True
                break
        if not clone_ok:
            raise RuntimeError("Failed to clone bootcamp repository in Colab.")

    candidate_tutorial_dirs = [
        Path("/content/ai4chemistry-bootcamp/tutorials"),
        Path("/content/ai4chemistry-bootcamp/website/tutorials"),
        Path.cwd(),
    ]
    tutorial_dir = _find_tutorial_dir(candidate_tutorial_dirs)

if tutorial_dir is None:
    raise FileNotFoundError(
        "Could not find neural-networks-mlp/dataset.csv. "
        "Set your working directory to the tutorials folder containing neural-networks-mlp/. "
        f"Current directory: {Path.cwd()}"
    )

# Every relative path later in the notebook (PathsConfig) assumes this directory.
os.chdir(tutorial_dir)
print(f"Working directory set to: {Path.cwd()}")
print(f"Found data root at: {Path.cwd() / 'neural-networks-mlp'}")

In [ ]:
# --- Cell: configuration and file-existence check ---
# Two small config objects keep "where are the files" (PathsConfig) separate from
# "how should this run behave" (TutorialConfig), so you only need to touch one or
# the other depending on what you're changing.

@dataclass
class PathsConfig:
    """Paths to every file this notebook reads or writes, relative to the
    tutorial directory set up in the previous cell."""
    dataset_csv: str = "./neural-networks-mlp/dataset.csv"
    train_split_csv: str = "./neural-networks-mlp/splits/train.csv"
    val_split_csv: str = "./neural-networks-mlp/splits/val.csv"
    test_split_csv: str = "./neural-networks-mlp/splits/test.csv"
    schema_csv: str = "./neural-networks-mlp/leaderboard/results_schema.csv"
    helper_py: str = "./neural-networks-mlp/leaderboard/submit_payload.py"

@dataclass
class TutorialConfig:
    """Settings that control how the notebook runs, independent of file paths.
    `target_col` picks which of the 12 Tox21 assays is the running example for
    Exercises 4-9 — all 12 are loaded and stay available in `df` (see Exercise 1),
    so you can change this to explore a different assay. `student_or_team` is
    used to tag your leaderboard submission — edit it before running the final cells."""
    target_col: str = "SR-MMP"
    fp_radius: int = 2
    fp_n_bits: int = 2048
    split_version: str = "v1"
    metric_name: str = "roc_auc"
    notebook_version: str = "nn_tutorial_v2_tox21"
    student_or_team: str = "Team XX"

PATHS = PathsConfig()
CFG = TutorialConfig()

def validate_required_files(paths: PathsConfig):
    """Fail early with a clear message if any required data/leaderboard file is
    missing, instead of letting a cryptic error surface deep in the notebook."""
    required = [
        paths.dataset_csv,
        paths.train_split_csv,
        paths.val_split_csv,
        paths.test_split_csv,
        paths.schema_csv,
        paths.helper_py,
    ]
    missing = [p for p in required if not Path(p).exists()]
    if missing:
        raise FileNotFoundError(
            "Missing required tutorial files:\n"
            + "\n".join(f"- {p}" for p in missing)
            + f"\n\nCurrent working directory: {Path.cwd()}"
        )

validate_required_files(PATHS)
print(PATHS)
print(CFG)

---
## Exercise 1 — Introduce the Tox21 dataset

**Tox21** ("Toxicology in the 21st Century") is a US EPA/NIH/FDA screening effort that tested ~8,000 compounds against 12 assays for nuclear-receptor signaling and cellular stress response (e.g. `NR-AR` = androgen receptor activation, `SR-p53` = p53 stress-response pathway). Each assay is a separate binary classification task: *does this compound trigger this specific biological response?*

Crucially, **not every compound was tested against every assay** — the label matrix is sparse. This matters both for picking a single "primary" task to train on (Exercises 3-9) and for the multi-task model later (Exercise 10), which has to learn from a matrix full of holes.

1. Load `dataset.csv` and look at its shape and columns — how many molecules, and what does each of the 12 task columns contain?
2. For each of the 12 tasks, compute how many molecules have a label at all, and what fraction of those are positive (toxic).
3. Visualize the missing-label pattern directly: for a sample of molecules, which of the 12 assays were they actually tested on?

In [ ]:
# --- Exercise 1: load the dataset and inspect its shape/columns ---
# your code here!
#
# 1. Read PATHS.dataset_csv into a DataFrame `df`.
# 2. Build a list `TOX21_TASKS` of the 12 task-label column names — every column
#    of `df` except "sample_id" and "smiles".
# 3. Print how many molecules and how many tasks there are, and show df.head().

In [ ]:
# --- Exercise 1: per-task label counts and positive rates ---
# your code here!
#
# Write plot_label_stats(df, tasks) that:
#   - counts how many non-missing labels each task has: df[tasks].notna().sum()
#   - computes each task's positive rate among labeled molecules:
#       (df[task] == 1).sum() / df[task].notna().sum()
#   - plots both as a pair of side-by-side bar charts
#   - returns (counts, positive_rates)
# Then call it as plot_label_stats(df, TOX21_TASKS) and print the resulting table.

def plot_label_stats(df, tasks):
    """Your turn — see the hint above."""
    raise NotImplementedError

label_counts, positive_rates = plot_label_stats(df, TOX21_TASKS)

In [ ]:
# --- Exercise 1: missing-label heatmap ---
# your code here!
#
# Write plot_missingness(df, tasks, n_show=300, seed=SEED) that:
#   - builds a 0/1 matrix from df[tasks].notna() (1 = tested, 0 = missing)
#   - randomly subsamples n_show molecules with a np.random.RandomState(seed)
#     (a full ~7800-row heatmap is unreadable)
#   - shows it with plt.imshow(..., cmap="Greys"), with the 12 tasks labeled on the x-axis
# Then call it on (df, TOX21_TASKS), and print label_counts/positive_rates for CFG.target_col.

def plot_missingness(df, tasks, n_show=300, seed=SEED):
    """Your turn — see the hint above."""
    raise NotImplementedError

plot_missingness(df, TOX21_TASKS)

---
## Exercise 2 — Visualize and choose a featurization

An MLP needs a fixed-length numeric vector per molecule, not a SMILES string. We'll use **Morgan fingerprints** (a.k.a. ECFP): for each atom, hash its local neighborhood (up to `radius` bonds away) into one of `n_bits` positions — the result is a fixed-length bit vector that captures which substructures are present, without needing 3D structure or a trained embedding model.

1. Compute a Morgan fingerprint matrix (`radius=2`, `n_bits=2048`) for every molecule's SMILES.
2. Project it to 2D with PCA and plot it, colored by the primary task's label (`CFG.target_col`) — do toxic and non-toxic molecules separate at all in fingerprint space?
3. Sanity-check the featurization quantitatively: does a simple k-NN classifier trained directly on the fingerprints do better than chance at predicting the label?
4. Attach the resulting `fp_0..fp_{n_bits-1}` columns to `df` so later exercises can use them as model input.
5. Look at the actual chemistry behind the labels: draw ~10 molecules positively labeled for `CFG.target_col` next to ~10 negatively labeled ones.
6. Quantify chemical similarity with **Tanimoto similarity** (intersection over union of the fingerprint bits that are set): compute the average pairwise similarity *within* the positively-labeled molecules, and *between* the positive and negative classes. **Discuss**: do positively-labeled molecules resemble each other more than they resemble negatives? What would that imply about how learnable this task is for a model working only from fingerprints?

In [ ]:
# --- Exercise 2: Morgan fingerprints from SMILES ---
# your code here!
#
# Write morgan_matrix(smiles_list, radius=2, n_bits=2048) that, for each SMILES:
#   - parses it with Chem.MolFromSmiles
#   - builds a Morgan fingerprint generator: rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
#   - gets the fingerprint and converts it to a numpy array with DataStructs.ConvertToNumpyArray
#   - falls back to a zero vector for unparseable SMILES
# and returns an (n_molecules, n_bits) array. Then compute X_fp for df["smiles"].
#
# Hint: this is adapted from `morgan_matrix` in molecular-representations.ipynb (Tutorial 1) —
# take a look there if you get stuck.

def morgan_matrix(smiles_list, radius=2, n_bits=2048):
    """Your turn — see the hint above."""
    raise NotImplementedError

X_fp = morgan_matrix(df["smiles"], radius=CFG.fp_radius, n_bits=CFG.fp_n_bits)

In [ ]:
# --- Exercise 2: chemical-space PCA plot, colored by the primary task ---
# your code here!
#
# Write plot_chemical_space(X, labels, title) that:
#   - projects X to 2D with sklearn's PCA (np.nan_to_num the input first)
#   - scatters untested molecules (labels is NaN) in light gray
#   - scatters tested molecules colored by their label (e.g. cmap="coolwarm"), with a colorbar
#
# Hint: adapted from `project_2d`/`plot_chemical_space` in molecular-representations.ipynb.

def plot_chemical_space(X, labels, title):
    """Your turn — see the hint above."""
    raise NotImplementedError

plot_chemical_space(X_fp, df[CFG.target_col].values, title=f"Morgan fingerprint chemical space, colored by {CFG.target_col}")

In [ ]:
# --- Exercise 2: quick k-NN quality check for this featurization ---
# your code here!
#
# Write evaluate_representation_knn(X, y, k=5, test_size=0.2, random_state=SEED) that:
#   - drops molecules with a missing label (np.isnan(y))
#   - splits the rest into train/test with train_test_split(..., stratify=y)
#   - standardizes features: fit a StandardScaler on the train split ONLY, then
#     reuse that fit to transform the test split (fitting fresh on test would leak
#     the test set's own distribution into how it gets scaled)
#   - fits a KNeighborsClassifier(n_neighbors=k) and reports test ROC-AUC

def evaluate_representation_knn(X, y, k=5, test_size=0.2, random_state=SEED):
    """Your turn — see the hint above."""
    raise NotImplementedError

_ = evaluate_representation_knn(X_fp, df[CFG.target_col].values)

In [ ]:
# --- Exercise 2: attach the fingerprint columns to df ---
# your code here!
#
# Build fp_cols = ["fp_0", ..., "fp_{n_bits-1}"], then pd.concat a DataFrame built
# from X_fp (using those column names) onto df. Later exercises (starting with
# Exercise 4's load_fixed_splits) look for columns prefixed with "fp_" as model input.

In [ ]:
# --- Exercise 2: draw positively vs. negatively labeled molecules ---
# your code here!
#
# Sample ~10 SMILES with df[CFG.target_col] == 1 and ~10 with df[CFG.target_col] == 0
# (df.loc[..., "smiles"].sample(n=10, random_state=SEED)), parse each with
# Chem.MolFromSmiles, and render each group with Draw.MolsToGridImage (wrap in
# display(...) since we call it twice in one cell).

In [ ]:
# --- Exercise 2: pairwise Tanimoto similarity, within vs. across classes ---
# your code here!
#
# Write tanimoto_matrix(A, B=None): pairwise Tanimoto similarity (intersection over
# union) between binary fingerprint rows of A (and B, or A vs. itself). Vectorize it
# with a matrix multiply: intersection = A @ B.T; union = a_sum[:,None] + b_sum[None,:] - intersection.
#
# Then compute: (a) the average pairwise similarity within X_fp rows where
# CFG.target_col == 1 (exclude self-pairs via np.triu_indices_from(..., k=1)), and
# (b) the average pairwise similarity between those positive rows and a size-matched
# random sample of CFG.target_col == 0 rows. Print both.

def tanimoto_matrix(A, B=None):
    """Your turn — see the hint above."""
    raise NotImplementedError

---
## Exercise 3 — Introduce the neural network

### 3a. The simplest possible network

Before building a configurable model, look at the simplest neural network that improves on a plain linear model — a single hidden layer, exactly like the introductory example in dmol.pub's *Deep Learning for Molecules and Materials* (Section C.6, https://dmol.pub/dl/introduction.html). It replaces the raw feature vector with **one trainable nonlinear transformation** g(x) = tanh(W0 x + b0), and then reads out a single value with a linear layer:

`Input -> Dense(D -> 32) -> Tanh -> Dense(32 -> 1) -> Output`

1. Build this exact architecture with `nn.Sequential` and run it on a small batch of Morgan fingerprints from Exercise 2, just to confirm the input/output shapes make sense (we train it for real using the machinery built in Exercise 4).
2. This network is fixed: exactly **one** hidden layer, exactly **32** units, always `tanh`, no regularization. Exercises 5-10 need to apply regularization (Exercise 5), compare depth/width (Exercise 6), compare other hyperparameters (Exercises 7-8), and eventually predict several tasks at once (Exercise 10) — none of that is possible if those choices are hard-coded. What would need to change to make this configurable?
3. Before wrapping training into a reusable function (Exercise 4), write out **one manual optimization cycle** as plain, top-level code — the same five-step pattern (zero the gradients, forward pass, compute the loss, backpropagate, step the optimizer) that every training loop in this notebook is built from, made fully visible here with nothing hidden inside a function.

### 3b. Generalizing it: `MolecularMLP`

`MolecularMLP` below is exactly that generalization: the same `Linear -> activation -> (Dropout)` block from 3a, just repeated `hidden_layers` times with a configurable `hidden_width`, a choice of `activation`, optional `dropout` for regularization, and a configurable number of outputs (`out_dim`, used for multi-task learning in Exercise 10).

4. Read through `MolecularMLP` — what does each constructor argument (`hidden_layers`, `hidden_width`, `dropout`, `activation`) control, and how does setting `hidden_layers=1, hidden_width=32, dropout=0, activation="tanh"` recover the simple network from 3a?
5. Instantiate a small one and check `count_params` — how many trainable parameters does a `hidden_layers=2, hidden_width=128` network have, versus `hidden_layers=4, hidden_width=512`? Keep this number in mind for Exercise 6.

In [ ]:
# --- Exercise 3a: the simplest possible network (dmol.pub-style) ---
# your code here!
#
# Build `simple_net` with nn.Sequential: Linear(CFG.fp_n_bits -> 32), Tanh(), Linear(32 -> 1),
# moved `.to(DEVICE)`. Then run it on a small batch (e.g. the first 8 rows of X_fp,
# as a float32 tensor moved to DEVICE) and print the input/output shapes — no
# training yet, just a forward-pass shape check.

simple_net = None  # your code here!

In [ ]:
# --- Exercise 3a: one manual optimization cycle, spelled out ---
# your code here!
#
# Build a small labeled batch: xb_demo/yb_demo from the first 64 rows of X_fp/
# df[CFG.target_col] where the label isn't missing (as float32 tensors on DEVICE,
# yb_demo needs an extra trailing dimension via .unsqueeze(1)).
#
# Define criterion = nn.BCEWithLogitsLoss() and optimizer = torch.optim.Adam(simple_net.parameters(), lr=1e-2).
#
# Then write a plain `for step in range(20):` loop — NOT inside a function — doing,
# in order: optimizer.zero_grad(), a forward pass (logits = simple_net(xb_demo)),
# loss = criterion(logits, yb_demo), loss.backward(), optimizer.step(). Print the
# loss each step and confirm it trends downward.

labeled_mask = df[CFG.target_col].notna().values

In [ ]:
# --- Exercise 3b: generalizing simple_net into a configurable MLP ---
# your code here!
#
# Write class MolecularMLP(nn.Module), generalizing simple_net (3a) into:
#   __init__(self, in_dim, hidden_layers, hidden_width, dropout, activation, out_dim=1):
#     - map "relu"/"gelu"/"tanh" strings to their nn.Module class (raise ValueError otherwise)
#     - stack `hidden_layers` blocks of [Linear(prev_dim, hidden_width), activation(), (Dropout(dropout) if dropout > 0)]
#     - end with a final Linear(prev_dim, out_dim) — this is what `out_dim` controls
#     - wrap everything in self.net = nn.Sequential(*layers)
#   forward(self, x): return self.net(x)
#
# Also write count_params(model): sum(p.numel() for p in model.parameters() if p.requires_grad)
#
# Then instantiate a hidden_layers=2, hidden_width=128 model and a hidden_layers=4,
# hidden_width=512 model (both dropout=0.1, activation="relu") and print their param counts.

class MolecularMLP(nn.Module):
    """Your turn — see the hint above."""
    def __init__(self, in_dim, hidden_layers, hidden_width, dropout, activation, out_dim=1):
        super().__init__()
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

def count_params(model):
    """Your turn — see the hint above."""
    raise NotImplementedError

---
## Exercise 4 — Train a model on one Tox21 task

Time to put the pieces together: load the fixed train/val/test split for `CFG.target_col`, build the training loop (forward pass, loss, backward pass, optimizer step), and train one baseline configuration end to end.

Recall the **fixed split policy**: `splits/train.csv`, `val.csv`, `test.csv` were published once by the instructor (see `neural-networks-mlp/splits/README.md`) — never re-split the data yourself, and never use the test split for anything except a final, one-time evaluation.

1. Load the fixed splits for `CFG.target_col`, dropping molecules that were never tested on it.
2. Train a single baseline MLP configuration and plot its training/validation loss and validation ROC-AUC over epochs.
3. Report its test ROC-AUC — this is the number every later exercise tries to beat.

In [ ]:
# --- Exercise 4: load the fixed split for the primary task ---
# your code here!
#
# Write _read_split_ids(path): read a split CSV and return its "sample_id" column
# as a set of strings.
#
# Write load_fixed_splits(df, train_csv, val_csv, test_csv, target_col):
#   - copy df, cast sample_id to str, and df.dropna(subset=[target_col])
#   - read train/val/test id sets with _read_split_ids and check they're disjoint
#   - feature_cols = columns of df starting with "fp_" (raise if none found)
#   - slice df into train/val/test DataFrames by sample_id membership (raise if any is empty)
#   - build x_{train,val,test} from feature_cols and y_{train,val,test} from target_col
#   - fit a StandardScaler on x_train only, then reuse that fit (not a fresh one) to
#     transform x_val/x_test — fitting fresh on val/test would leak their own
#     distributions into how they get scaled
#   - return {"feature_cols": ..., "train": (x, y, ids), "val": (...), "test": (...)}
#
# Then call it with (df, PATHS.train_split_csv, PATHS.val_split_csv, PATHS.test_split_csv,
# CFG.target_col) and unpack x_train/y_train/train_ids etc.

def _read_split_ids(path):
    """Your turn — see the hint above."""
    raise NotImplementedError

def load_fixed_splits(df, train_csv, val_csv, test_csv, target_col):
    """Your turn — see the hint above."""
    raise NotImplementedError

In [ ]:
# --- Exercise 4: tensors and DataLoaders ---
# your code here!
#
# Convert x_train/x_val/x_test and y_train/y_val/y_test to float32 torch tensors
# (the y tensors need an extra trailing dimension: .unsqueeze(1)).
#
# Write make_loaders(batch_size) that wraps each (x, y) pair in a TensorDataset and
# a DataLoader, shuffling only the training loader, and returns (train_loader, val_loader, test_loader).

def make_loaders(batch_size):
    """Your turn — see the hint above."""
    raise NotImplementedError

In [ ]:
# --- Exercise 4: one training epoch, evaluation, and the full training loop ---
# your code here!
#
# Write train_one_epoch(model, loader, criterion, optimizer): for each (xb, yb)
# batch, move to DEVICE, zero_grad, forward pass, compute loss, loss.backward(),
# optimizer.step(), accumulate loss.item() * xb.size(0). Return the average loss
# over len(loader.dataset).
#
# Write evaluate(model, loader, criterion) (decorate with @torch.no_grad()): same
# loop in eval mode, collect sigmoid(logits) as predicted probabilities and the
# true labels, and return (avg_loss, roc_auc_score(y_true, y_pred), y_true, y_pred)
# as numpy arrays.
#
# Write run_experiment(config, split=None, verbose=False): if split is None, build
# loaders with make_loaders(config["batch_size"]) and use x_train.shape[1] as
# in_dim; otherwise build DataLoaders directly from split["train"]/["val"]/["test"]
# (each an (x_tensor, y_tensor) pair — Exercise 6's variability sub-exercise passes
# this to retrain on ad hoc splits) and take in_dim from split["train"][0].shape[1].
# Build a MolecularMLP from config's hidden_layers/hidden_width/dropout/activation, a
# BCEWithLogitsLoss criterion, and an Adam or SGD optimizer per config["optimizer"].
# Loop over config["epochs"]: train_one_epoch, evaluate on val, print the epoch's
# train/val loss and val score when verbose, checkpoint the best-so-far weights
# (higher val ROC-AUC is better), and stop early after config["early_stop_patience"]
# epochs without improvement. Reload the best checkpoint, evaluate once on the test
# loader, and return a dict with model, history, train_time_sec, best_val_score,
# test_score, test_loss, y_true_test, y_pred_test, epochs_trained, param_count.

def train_one_epoch(model, loader, criterion, optimizer):
    """Your turn — see the hint above."""
    raise NotImplementedError

def evaluate(model, loader, criterion):
    """Your turn — see the hint above."""
    raise NotImplementedError

def run_experiment(config):
    """Your turn — see the hint above."""
    raise NotImplementedError

In [ ]:
# --- Exercise 4: plot learning curves, and train the baseline model ---
# your code here!
#
# Write plot_learning_curves(history, title): a 1x2 figure — left: history["train_loss"]
# and history["val_loss"] vs. epoch; right: history["val_score"] vs. epoch.
#
# Then run_experiment(baseline_config, verbose=True) with the baseline_config below
# (verbose=True prints train/val loss each epoch, while training, before the plot),
# plot its history, and print its test ROC-AUC and parameter count.

baseline_config = {
    "hidden_layers": 2, "hidden_width": 128, "activation": "relu", "dropout": 0.1,
    "optimizer": "adam", "learning_rate": 1e-3, "batch_size": 128, "weight_decay": 1e-4,
    "epochs": 40, "early_stop_patience": 8,
}

def plot_learning_curves(history, title):
    """Your turn — see the hint above."""
    raise NotImplementedError

---
## Exercise 5 — Regularization

Look back at the baseline learning curve from Exercise 4: training loss keeps dropping while validation loss plateaus or rises — a classic overfitting signature (and if you ran it with `verbose=True`, you'll have seen the per-epoch numbers tell the same story before the plot even appeared). Per dmol.pub's overview of regularization ([Section 7.6](https://dmol.pub/dl/layers.html) — early stopping, weight decay, activity regularization, batch normalization, dropout), there are several standard ways to fight this by constraining the model, rather than only gathering more data. `run_experiment` already exposes three of them as config knobs:

- **Early stopping** — already running: `run_experiment` checkpoints the best validation score and stops once `early_stop_patience` epochs pass without improvement. Tightening `early_stop_patience` makes this constraint stronger (it stops sooner, before the train/val gap can grow as large).
- **Weight decay (L2)** — the `weight_decay` value passed to the optimizer adds a penalty proportional to the squared magnitude of the weights, discouraging any single weight from growing large enough to memorize noise.
- **Dropout** — the `dropout` value randomly zeroes a fraction of hidden units on each training step, forcing the network to spread information across redundant "pathways" rather than relying on any one unit ([Srivastava et al., 2014](https://jmlr.org/papers/v15/srivastava14a.html)).

1. Starting from Exercise 4's `baseline_config`, create three variants that each change **only one** regularization knob: a stronger `dropout` (0.5), a stronger `weight_decay` (1e-2), and a tighter `early_stop_patience` (2).
2. Train all three (reusing the original `baseline_result` from Exercise 4 rather than retraining it) and plot each variant's learning curve — which one shrinks the train/val loss gap the most?
3. Compare test ROC-AUC across all four runs in a summary table. **Discuss**: does the regularizer that reduces overfitting the most also give the best (or worst) test score? What does that imply about the cost/benefit of each knob, and would combining them help further?

In [ ]:
# --- Exercise 5: three single-knob regularization variants vs. the Exercise 4 baseline ---
# your code here!
#
# Build regularization_variants: a dict mapping a name to a config — the baseline
# (reuse baseline_config as-is), "stronger dropout" ({**baseline_config, "dropout": 0.5}),
# "stronger weight_decay" ({**baseline_config, "weight_decay": 1e-2}), and "tighter
# early stopping" ({**baseline_config, "early_stop_patience": 2}).
#
# For each variant: reuse baseline_result for the baseline entry (don't retrain it),
# call run_experiment(cfg) for the other three, and plot_learning_curves(result["history"], title=name)
# for all four. Collect a row per variant with variant/val_score/test_score/
# train_loss/val_loss/train_val_loss_gap (same gap computation as Exercise 6's
# capacity sweep: history["val_loss"][-1] - history["train_loss"][-1]) into a
# regularization_df.

regularization_variants = {
    "baseline (Ex.4)": baseline_config,
    "stronger dropout": {**baseline_config, "dropout": 0.5},
    "stronger weight_decay": {**baseline_config, "weight_decay": 1e-2},
    "tighter early stopping": {**baseline_config, "early_stop_patience": 2},
}

---
## Exercise 6 — Question the NN architecture in light of the dataset size

### 6a. Capacity vs. dataset size

`load_fixed_splits` reported the training set size for `CFG.target_col` above — likely a few thousand molecules. A `hidden_layers=4, hidden_width=512` MLP has hundreds of thousands of parameters (Exercise 3). When the parameter count starts to rival (or exceed) the number of training examples, a model can memorize the training set instead of generalizing — the same overfitting signature Exercise 5 just fought with regularization, but here driven by capacity instead.

1. Train three MLPs that only differ in capacity (`tiny` / `medium` / `large` — same everything else, and the same *unregularized* defaults as Exercise 4's baseline), and record their parameter count, val/test ROC-AUC, and final train/val loss alongside the loss gap.
2. Plot parameter count against test score, and against the train/val loss gap — does a bigger model actually help here, or does it just overfit faster?

### 6b. How much does a single run's score vary?

A single `test_score` per config could just be luck — a different random weight initialization (and batch shuffling order), or a different train/val/test split, might give a noticeably different number. Quantify **both** sources of variability separately, using the same tiny/medium/large configs (fast to train, so repeating them is cheap):

3. **Weight-init/shuffling variability**: keeping the official fixed split, retrain each capacity config 5 times with a different `torch.manual_seed(...)` before each `run_experiment` call. Record the spread (std) of `test_score` across repeats.
4. **Split variability**: keeping the same model configs, instead build 5 *fresh*, one-off random train/val/test splits (stratified on `CFG.target_col`, never the official fixed split) with different seeds, and retrain each config once per split. Record the spread (std) across these differently-split repeats. This is a diagnostic only — the leaderboard submission (Exercise 9) always uses the official fixed split from `splits/*.csv`.
5. Plot capacity vs. test score with two sets of error bars, one per variability source — which source of variance is larger, and does that change how you read the Exercise 6a capacity-vs-performance plot?

In [ ]:
# --- Exercise 6: capacity sweep at fixed dataset size ---
# your code here!
#
# For each config in capacity_configs (merged with capacity_defaults), call
# run_experiment(...), compute the final-epoch train/val loss gap
# (history["val_loss"][-1] - history["train_loss"][-1]), and collect a row with
# label/hidden_layers/hidden_width/param_count/best_val_score/test_score/
# train_loss/val_loss/train_val_loss_gap. Build capacity_df from the collected rows.

capacity_configs = [
    {"label": "tiny",   "hidden_layers": 1, "hidden_width": 32},
    {"label": "medium", "hidden_layers": 2, "hidden_width": 128},
    {"label": "large",  "hidden_layers": 4, "hidden_width": 512},
]
capacity_defaults = {
    "activation": "relu", "dropout": 0.1, "optimizer": "adam", "learning_rate": 1e-3,
    "batch_size": 128, "weight_decay": 1e-4, "epochs": 40, "early_stop_patience": 8,
}

In [ ]:
# --- Exercise 6: parameter count vs. test score and overfitting gap ---
# your code here!
#
# Make a 1x2 figure from capacity_df: left — param_count vs. test_score; right —
# param_count vs. train_val_loss_gap (with a horizontal line at 0). Annotate each
# point with its "label" (tiny/medium/large).

In [ ]:
# --- Exercise 6b: weight-init/shuffling variability ---
# your code here!
#
# For each capacity config, repeat N_REPEATS=5 times: call torch.manual_seed(seed)
# (seed = 0..4) right before run_experiment(full_config) — no `split=` kwarg, so it
# still trains/evaluates on the official fixed split. Collect the 5 test_score
# values per config, then report their mean and std. Build init_variability_df from
# one row per config (label, mean_test_score, std_test_score).

N_REPEATS = 5
init_seeds = list(range(N_REPEATS))

In [ ]:
# --- Exercise 6b: split variability ---
# your code here!
#
# Write make_random_split(df, feature_cols, target_col, seed, val_frac=0.15, test_frac=0.15):
# a ONE-OFF diagnostic split (not the official fixed split) — stratified
# train_test_split the labeled rows of df into train/val/test, scale the same
# fit-train/transform-rest way as load_fixed_splits, convert to tensors, and
# return {"train": (x,y), "val": (x,y), "test": (x,y)}.
#
# For each capacity config, repeat N_REPEATS=5 times with a different seed: build
# a fresh custom_split via make_random_split(df, split_data["feature_cols"],
# CFG.target_col, seed), then run_experiment(full_config, split=custom_split).
# Collect the 5 test_score values per config, report mean/std, and build
# split_variability_df (one row per config).

split_seeds = list(range(N_REPEATS))

In [ ]:
# --- Exercise 6b: combined variability plot ---
# your code here!
#
# Plot capacity_df["param_count"] on the x-axis, with two overlaid plt.errorbar
# series — init_variability_df's mean/std and split_variability_df's mean/std —
# so both variability sources are visible on the same axes. Annotate points with
# each config's "label".

---
## Exercise 7 — Visualize learning curves with regard to hyperparameters

Beyond capacity, other hyperparameters change *how* training unfolds — not just the final score. The learning rate is a good one to isolate: too high and training can oscillate or diverge, too low and it may not converge within the epoch budget.

1. Fix the architecture at the `medium` configuration from Exercise 6.
2. Train three models that only differ in `learning_rate` (`1e-2`, `1e-3`, `1e-4`).
3. Overlay their validation ROC-AUC curves (and validation loss curves) on the same axes — what does each learning rate's curve shape tell you?

In [ ]:
# --- Exercise 7: overlaid learning curves across learning rates ---
# your code here!
#
# Write plot_overlaid_curves(histories, labels, key, ylabel, title): one line per
# (history, label) pair, plotting history[key] vs. epoch on shared axes.
#
# For each lr in lr_values, call run_experiment({**lr_defaults, "learning_rate": lr})
# and collect the results. Then call plot_overlaid_curves twice: once with
# key="val_score", once with key="val_loss".

lr_values = [1e-2, 1e-3, 1e-4]
lr_defaults = {
    "hidden_layers": 2, "hidden_width": 128, "activation": "relu", "dropout": 0.1,
    "optimizer": "adam", "batch_size": 128, "weight_decay": 1e-4,
    "epochs": 40, "early_stop_patience": 8,
}

def plot_overlaid_curves(histories, labels, key, ylabel, title):
    """Your turn — see the hint above."""
    raise NotImplementedError

---
## Exercise 8 — Run a hyperparameter search

Exercises 6 and 7 varied one hyperparameter at a time. Now run a small grid over several hyperparameters jointly, and compare runs the way you would for a real project: by validation score for model selection, and by training cost for the efficiency/performance tradeoff.

Each dict in `search_space` is one full config passed to `run_experiment`. Feel free to add, remove, or edit entries for your own experiments.

1. Run every configuration in `search_space` and collect a comparison table.
2. Select the best run **using the validation score only** — the test score is diagnostic, never used for selection.
3. Plot training time against test score for every run — which configurations sit on the efficiency/performance frontier?

In [ ]:
# --- Exercise 8: a small hyperparameter grid ---
# your code here!
#
# For each config in search_space, call run_experiment, and collect a row with the
# swept hyperparameters plus epochs_trained/param_count/train_time_sec/best_val_score/
# test_score (keep "history"/"model"/"y_pred_test"/"y_true_test" too — later cells
# need them). Build runs_df from the collected rows (drop the heavy fields for the
# DataFrame).

search_space = [
    {"hidden_layers": 2, "hidden_width": 128, "activation": "relu", "dropout": 0.1, "optimizer": "adam", "learning_rate": 1e-3, "batch_size": 128, "weight_decay": 1e-4, "epochs": 40, "early_stop_patience": 8},
    {"hidden_layers": 3, "hidden_width": 256, "activation": "relu", "dropout": 0.2, "optimizer": "adam", "learning_rate": 1e-3, "batch_size": 128, "weight_decay": 1e-4, "epochs": 40, "early_stop_patience": 8},
    {"hidden_layers": 4, "hidden_width": 256, "activation": "gelu", "dropout": 0.3, "optimizer": "adam", "learning_rate": 5e-4, "batch_size": 256, "weight_decay": 5e-4, "epochs": 50, "early_stop_patience": 10},
    {"hidden_layers": 2, "hidden_width": 64,  "activation": "tanh", "dropout": 0.0, "optimizer": "sgd",  "learning_rate": 1e-2, "batch_size": 64,  "weight_decay": 0.0,  "epochs": 50, "early_stop_patience": 10},
]

In [ ]:
# --- Exercise 8: pick the best run by VALIDATION score ---
# your code here!
#
# Find the row index of the highest "best_val_score" in runs_df (never use
# "test_score" for this selection), look up the matching entry in all_runs, and
# plot its learning curves.

In [ ]:
# --- Exercise 8: efficiency vs. performance (Pareto-front context) ---
# your code here!
#
# Scatter-plot runs_df["train_time_sec"] vs. runs_df["test_score"], annotating
# each point with its run index and architecture (hidden_layers/hidden_width).

---
## Exercise 9 — Conclusions

1. Summarize the best run's hyperparameters and scores, and connect it back to Exercise 6: does the winning configuration's size make sense given how much labeled data was available for `CFG.target_col`?
2. Plot the ROC curve for the best model's test predictions, alongside the scalar ROC-AUC already reported.
3. Submit the final run to the class leaderboard for the Pareto-front analysis.

In [ ]:
# --- Exercise 9: summary + ROC curve for the best model ---
# your code here!
#
# Print the best run's hyperparameters and val/test ROC-AUC (from runs_df/best_run).
# Then use sklearn's roc_curve(best_run["y_true_test"], best_run["y_pred_test"]) to
# get (fpr, tpr, thresholds), and plot fpr vs. tpr alongside the y=x "chance" diagonal.

In [ ]:
# --- Exercise 9: build the standardized leaderboard payload ---
# your code here!
#
# Write utc_now_iso(): current UTC time as an ISO-8601 string ending in "Z"
# (datetime.now(timezone.utc), drop microseconds, replace "+00:00" with "Z").
#
# Build `payload`: a dict with exactly the fields listed in
# neural-networks-mlp/leaderboard/results_schema.csv — run_id, timestamp_utc,
# student_or_team, dataset_name="tox21", task_type="binary_classification",
# split_version, model_family="MLP", the best_run's hyperparameters and scores,
# metric_name, device, seed, notebook_version, and a notes string. Print it with
# json.dumps(payload, indent=2).

def utc_now_iso():
    """Your turn — see the hint above."""
    raise NotImplementedError

In [ ]:
# --- Exercise 9: write local submission artifacts ---
# your code here!
#
# Load neural-networks-mlp/leaderboard/submit_payload.py as a module (see
# importlib.util.spec_from_file_location / module_from_spec / spec.loader.exec_module),
# then use its write_payload_json(payload, json_path) and
# append_payload_csv(payload, csv_log_path) to write your submission under
# neural-networks-mlp/leaderboard/submissions/.

In [ ]:
# --- Exercise 9: optional — submit to a shared leaderboard endpoint ---
# your code here!
#
# If APPS_SCRIPT_URL is set, POST `payload` as JSON to it with `requests` and
# print the response status/text. Otherwise, skip (print that it was skipped).

APPS_SCRIPT_URL = ""

---
## Exercise 10 (Advanced) — Multi-task Tox21

So far each model only ever saw one Tox21 assay. But the 12 assays share underlying chemistry (e.g. several are nuclear-receptor pathways) — a model with a shared hidden trunk and one output head per task can potentially learn a better shared representation, especially useful when any single task's labeled set is small.

The catch: the label matrix is sparse (Exercise 1), so the loss must **mask out untested (task, molecule) pairs** rather than treating a missing label as a real 0.

1. Build `(X, Y, W)` for each fixed split, over *all* molecules regardless of `CFG.target_col` — `Y` is the 12-task label matrix (missing filled with 0) and `W` is a same-shaped mask (1 where a label exists, 0 otherwise).
2. Reuse `MolecularMLP` with `out_dim=len(TOX21_TASKS)`, and write a masked version of the training/evaluation loop: `BCEWithLogitsLoss(reduction="none")` multiplied by the mask, averaged only over valid labels.
3. Train it, report a per-task test ROC-AUC table plus the mean across tasks, and compare the shared `CFG.target_col` task's multi-task score against its Exercise 8 single-task score.

In [ ]:
# --- Exercise 10: build multi-task (X, Y, W) arrays over ALL molecules ---
# your code here!
#
# Write build_multitask_split(df, tasks, sample_ids): slice df to the given
# sample_ids, build X from fp_cols, y_raw from the `tasks` columns (with NaN for
# missing labels), W = ~np.isnan(y_raw) as float, Y = np.nan_to_num(y_raw, nan=0.0).
# Return (X, Y, W).
#
# Read the full (unfiltered) train/val/test id sets with _read_split_ids, build
# each split's (X, Y, W) with build_multitask_split, then fit a StandardScaler on
# the train X only and transform all three.

def build_multitask_split(df, tasks, sample_ids):
    """Your turn — see the hint above."""
    raise NotImplementedError

In [ ]:
# --- Exercise 10: tensors, loaders, and the masked training/eval loop ---
# your code here!
#
# Build mt_train_loader/mt_val_loader/mt_test_loader: TensorDataset(X, Y, W) triples
# wrapped in DataLoaders (only the train loader shuffles).
#
# Write train_one_epoch_multitask(model, loader, optimizer): for each (xb, yb, wb)
# batch, compute BCEWithLogitsLoss(reduction="none")(logits, yb) * wb, take
# loss_elem.sum() / wb.sum().clamp(min=1) as the scalar loss to backprop, and
# return the mask-weighted average loss over the whole loader.
#
# Write evaluate_multitask(model, loader, tasks) (decorate with @torch.no_grad()):
# collect sigmoid(logits), y, and w over the whole loader, then for each task i
# compute roc_auc_score using only rows where w[:, i] > 0 (skip if fewer than 2
# such rows or the labels have no variation). Return (per_task_auc dict, mean_auc).

def train_one_epoch_multitask(model, loader, optimizer):
    """Your turn — see the hint above."""
    raise NotImplementedError

def evaluate_multitask(model, loader, tasks):
    """Your turn — see the hint above."""
    raise NotImplementedError

In [ ]:
# --- Exercise 10: train the multi-task model ---
# your code here!
#
# Build mt_model = MolecularMLP(..., out_dim=len(TOX21_TASKS)) and an Adam optimizer.
# Loop over epochs: train_one_epoch_multitask, evaluate_multitask on val, checkpoint
# the best-so-far weights by mean val ROC-AUC, and stop early after mt_patience
# epochs without improvement (same early-stopping pattern as run_experiment in
# Exercise 4). Reload the best checkpoint at the end.

mt_epochs, mt_patience = 40, 8

In [ ]:
# --- Exercise 10: per-task test ROC-AUC, and comparison to the single-task baseline ---
# your code here!
#
# Call evaluate_multitask(mt_model, mt_test_loader, TOX21_TASKS) to get per-task
# test ROC-AUC and the mean across tasks. Build a small DataFrame of per-task
# scores (sorted, most predictable task first) and print it, then print the mean.
# Finally, compare per_task_test_auc[CFG.target_col] (multi-task) against
# best_run["test_score"] (Exercise 8's single-task best) — did multi-task help or
# hurt this specific task?
#
# Note: this is exploratory — the shared leaderboard schema expects one score per
# submission, so this multi-task run isn't pushed to the same leaderboard as
# Exercise 9's primary single-task submission.

---
## Further Reading

- **Huang, Xia et al. (2016)** — *Tox21 Challenge to Build Predictive Models of Nuclear Receptor and Stress Response Pathways as Mediated by Exposure to Environmental Chemicals and Drugs* — Front. Environ. Sci. — overview of the Tox21 challenge and dataset.
- **Mayr et al. (2016)** — *DeepTox: Toxicity Prediction using Deep Learning* — Front. Environ. Sci. 3, 80 — the original deep multi-task model benchmarked on Tox21.
- **Rogers & Hahn (2010)** — *Extended-Connectivity Fingerprints* — J. Chem. Inf. Model. 50, 742 — the ECFP/Morgan fingerprint reference.
- **Srivastava et al. (2014)** — *Dropout: A Simple Way to Prevent Neural Networks from Overfitting* — J. Mach. Learn. Res. 15, 1929 — the dropout reference used in Exercise 5.
- **Ramsundar et al. (2015)** — *Massively Multitask Networks for Drug Discovery* — arXiv:1502.02072 — motivates the multi-task extension in Exercise 10.
- **[Molecular Toxicity Prediction using ML](https://www.kaggle.com/code/gyanendrachaubey/molecular-toxicity-prediction-using-ml)** (Kaggle, gyanendrachaubey) — visualization inspiration for this notebook's EDA and evaluation plots.